# Layer 3 · Step 2a: a pix2pix generator, trained, then run on the validation after-photos

Step 2a of `Essential_task_delegation/Layer3_Task/Layer3_Task_Brief.md`, built 24 Nov 2026 after Abrar's go. **GPU T4** (one is used). **Internet on**, to install `lpips` 0.1.4 and download its AlexNet weights.

**What it does, and why**
- It trains pix2pix (Isola et al., CVPR 2017) to turn an after-photo into a likely before-photo.
  - **Data:** only step 0's 5,798 training pairs (`pairs.csv`, split `train`). After-photo in, before-photo out.
  - **Crops:** a random 512 × 512 crop of the 1024 × 1024 photos. The same crop, rotation (0, 90, 180 or 270°) and flip go to both photos of a pair. No resizing.
  - **Networks,** written as in the authors' PyTorch code (`junyanz/pytorch-CycleGAN-and-pix2pix`, BSD licence), with its pix2pix defaults:
    - a U-Net generator (`unet_256`: 8 halvings, batch norm, dropout 0.5);
    - a 70 × 70 PatchGAN discriminator (3 layers) that sees the after-photo with a real or a generated before-photo;
    - the "vanilla" GAN loss.
  - **Settings:** L1 loss weight 100; Adam, lr 2e-4, beta1 0.5; batch size 1; seed 42.
  - **Learning rate:** 2e-4 for the first half of the epochs, then falling linearly toward 0, as the reference does (100 + 100 epochs there).
  - **Ours, not the reference's:** fp16 mixed precision, for speed on the T4.
- After training, it generates a before-photo for each of the 645 validation pairs, from the full 1024 × 1024 after-photo. The generator is fully convolutional, so it runs on the full size.
  - As in the paper and the reference's test script, the generator runs as in training: dropout on, batch norm with the picture's own statistics.
  - Each picture's dropout is seeded (42 + the pair's `pair_index` in `pairs.csv`), so the pictures can be made again.
- **Why:** step 2b runs `layer3-step1-baselines.ipynb` with these pictures in R11's before-photo slot, and judges them by the rule (brief, section 4).
- It saves:
  - the 645 pictures as PNG, named like the after-photos, in one zip;
  - a sheet of 12 samples from clean validation pairs: after-photo, generated, real before-photo. 2 come from Turkey, and 1 each from 10 other events, picked at random (seed 42), where possible among pairs with destroyed buildings;
  - how close each generated picture is to the real before-photo: PSNR, SSIM and LPIPS. The same is computed for the after-photo itself (`copy`), for comparison. The brief reports these beside the rule; they decide nothing;
  - the checkpoints, so a second session can continue, and step 5 can use the generator.

**Two sessions.** Kaggle stops a run after 12 hours.
- Each session trains whole epochs, and saves a checkpoint after every one. It starts no epoch that would end after 10.75 hours.
- Run again with this notebook's output attached as an input, it finds `layer3-step2a/checkpoints/last.pt` and continues from there. Nothing else changes.
- It generates the pictures only once all epochs are done.

**Why a zip.** Kaggle may limit a notebook's saved output to about 500 files; we could not confirm this. An 11-hour session must not fail when it saves. So the 645 PNGs are written to `/tmp` first, then packed, uncompressed, into `generated_before_photos.zip`.

**The smoke test** (`SMOKE_TEST = True`)
- 3 short epochs of 200 training pairs each, a new random 200 in each epoch.
- After epoch 2, it saves a checkpoint, loads it into new networks, and checks that every weight and optimizer value came back the same. Epoch 3 then trains from the loaded copy, as a second session would.
- It generates the 12 sample pairs only.
- From epochs 2 and 3, it measures the time per training pair. It then prints the number of epochs that fit in two sessions: EPOCHS = floor(20 hours / the time per epoch over 5,798 pairs), but at most 200, the reference's total. That number goes into Cell 2 for the full run.

**The checks.** The last cell prints them. If one fails, it stops with an error, and the printout goes to Abrar.
1. `pairs.csv` comes from step 0's full run (every check passed): 6,443 pairs, 5,798 training, 645 validation, 605 clean, every photo 1024 × 1024.
2. Training sees only training photos:
   - it reads only the 5,798 training rows, each once per epoch (smoke test: 200 per epoch);
   - no validation photo is among them, by name or by path;
   - by step 0's pixel hashes, no photo of a clean validation pair is among them. The 40 pairs that aren't clean share their before-photo with a training pair, as E5 found; that is why the rule is judged on the 605.
3. Training ran all its epochs, and every loss stayed finite. In the smoke test, the loaded checkpoint also matched what was saved.
4. All 645 pictures (smoke test: 12) are in the zip. Each is a 1024 × 1024 RGB PNG, named like its after-photo, and reads back exactly as generated, the way step 1 reads it.
5. The sample sheet holds 12 clean validation pairs from 11 events, 2 of them from Turkey.
6. PSNR, SSIM and LPIPS are finite for every pair. For `copy`, their means match the PC's numbers below: PSNR within 0.001, SSIM within 0.0001, LPIPS within 0.001.

A session that stops before the last epoch checks only 1 and 2. It ends by saying how many epochs are done.

**The expectation, fixed on 24 Nov before any run**
- **Smoke test:** all six checks pass.
  - Time per epoch: we guess 10 to 30 minutes, so EPOCHS between about 40 and 120.
  - The 12 sample pictures will be poor after 600 training steps. That's expected and decides nothing.
- **`copy`, computed on the PC** with Cell 7's code (the after-photo against the real before-photo):
  - on the 605 clean pairs: PSNR 18.2438 dB, SSIM 0.4588, LPIPS 0.4620;
  - on the 12 sample pairs: PSNR 16.7887 dB, SSIM 0.3773, LPIPS 0.4695.
  - The 12 sample pairs (event, then `pair_index`): turkey_earthquake 4657; turkey_earthquake 4694; beriut_explosion 31; hawaii_wildfire 65; hurricane_harvey 6197; kalehe_flooding 1193; midwest_flooding 6227; moore_tornado 1737; palu_tsunami 2227; portugal_wildfire 3422; santa_rosa_wildfire 3692; socal_fire 4060.
- **Full run:**
  - L1 loss falls over the epochs.
  - The pictures will be blurry. 52% of our pairs are shifted by more than 2 px between the two photos (Ridita's audit, `Task_Ledger.md` §9), and an L1 loss averages over the shift.
  - Where buildings were destroyed, the generated picture will mostly show the rubble restyled, not the lost building: the after-photo has no information about what stood there.
  - PSNR: the generated pictures closer to the real before-photo than `copy` is, because the L1 loss trains exactly that. SSIM and LPIPS: no prediction.
- **Our guess for step 2b,** written now and deciding nothing:
  - R11 with the generated pictures stays below the bar of 0.5604, and between `copy` (0.4046) and R9 (0.5402).
  - The reason: a picture made from the after-photo can't show what the destroyed buildings looked like.
  - On Turkey, no gain: even with the real before-photo, R11's destroyed IoU there is 0 (step 1).

**How to run on Kaggle**
1. A new notebook, `layer3-step2a-pix2pix-train`: File → Import Notebook → this file. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Inputs (+ Add Input):
   - the dataset `abrarmohammedtanzim/disasterm3-mirror`;
   - your step 0 notebook, `layer3-step0-pairs-and-checks`: the output of its full run.

   No checkpoints dataset is needed here.
3. Smoke test: `SMOKE_TEST = True`, then Save Version → Save & Run All. About 15 minutes.
4. After Abrar's go: set `SMOKE_TEST = False` and `EPOCHS` to the number the smoke test printed, then Save & Run All. Session 1 stops by itself after about 11 hours.
5. Session 2: add session 1's output as an input, change nothing else, and Save & Run All.
6. Outputs in `/kaggle/working/layer3-step2a/`:
   - small, for `results/step2a/`: `summary.json`, `train_log.csv`, `similarity_per_pair.csv`, `similarity.csv`, `samples.jpg`;
   - large, staying on Kaggle: `generated_before_photos.zip` (the 645 PNGs), `checkpoints/last.pt` (everything needed to continue) and `checkpoints/generator_final.pt` (the generator alone).

In [1]:
# ── Cell 1: Environment setup (E5 step 2's Cell 1; changed: the pin is lpips 0.1.4, installed without its dependencies, and no restart) ──
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "lpips": "0.1.4",   # the LPIPS picture distance (Zhang et al., CVPR 2018), reported beside the rule
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",   # --no-deps: Kaggle's torch stays as it is
                    *(f"{k}=={v}" for k, v in PINS.items())],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    print("✓ Dependencies installed. lpips was not imported before, so no restart is needed.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")

⏳ Installing pinned versions:
   lpips: not installed → 0.1.4
✓ Dependencies installed. lpips was not imported before, so no restart is needed.
  torch 2.11.0+cu130, python 3.13.16


In [2]:
# ── Cell 2: Configuration and inputs ──
import glob
import hashlib
import json
import multiprocessing
import os
import random
import time
from pathlib import Path

import numpy as np
import torch

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/layer3-step2a")    # saved as the notebook's output
TMP_DIR = Path("/tmp/layer3-step2a")               # not saved: the PNGs before they are zipped

SMOKE_TEST = False       # True = 3 short epochs of 200 pairs, then the 12 sample pictures (about 15 min); False = the full run
EPOCHS = 125          # the full run: the number the smoke test prints (training must fit in two 12-hour sessions)

# ── pix2pix: the defaults of the authors' code for pix2pix, except where marked ──
CROP = 512              # Abrar, 24 Nov: 512 px crops of the 1024 px photos (reference: 256 px crops of 286 px images)
BATCH_SIZE = 1          # reference
NGF, NDF = 64, 64       # reference: the generator's and the discriminator's widths
N_DOWNS = 8             # reference unet_256: 8 halvings
D_LAYERS = 3            # reference "basic" discriminator: the 70 x 70 PatchGAN
DROPOUT = True          # reference: dropout 0.5 in the generator's three innermost up-steps
LAMBDA_L1 = 100.0       # reference, and Abrar
LR, BETA1, BETA2 = 2e-4, 0.5, 0.999   # reference, and Abrar
AMP = True              # ours: fp16 mixed precision, for speed on the T4
SEED = 42               # the brief: seed 42 everywhere

# ── Time ──
SESSION_STOP_HOURS = 10.75    # a session starts no epoch that would end after this; Kaggle stops a run at 12 hours
TRAIN_HOURS = 20.0            # the epochs must fit in this: two sessions of about 10 hours (the smoke test's proposal)
EPOCHS_CAP = 200              # and the proposal is at most the reference's 100 + 100 epochs
SMOKE_EPOCHS, SMOKE_PAIRS_PER_EPOCH, SMOKE_RELOAD_AFTER = 3, 200, 2

# ── The sample sheet: 12 clean validation pairs, at random (seed 42) ──
SAMPLE_TURKEY, SAMPLE_OTHER_EVENTS = 2, 10       # 2 Turkey pairs, and 1 pair from each of 10 other events

# ── copy (the after-photo against the real before-photo), from the PC with Cell 7's code: the means (Cell 0) ──
COPY_EXPECTED = {"605": {"psnr": 18.243759, "ssim": 0.458838, "lpips": 0.461989},
                 "12": {"psnr": 16.788675, "ssim": 0.377303, "lpips": 0.469525}}
COPY_TOLERANCE = {"psnr": 0.001, "ssim": 0.0001, "lpips": 0.001}

if SMOKE_TEST:
    EPOCHS_RUN = SMOKE_EPOCHS
else:
    assert isinstance(EPOCHS, int) and EPOCHS >= 2, "Set EPOCHS in Cell 2 to the number the smoke test printed"
    EPOCHS_RUN = EPOCHS
NUM_WORKERS = min(4, os.cpu_count() or 1) if "fork" in multiprocessing.get_all_start_methods() else 0

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ── The inputs: the dataset and step 0's output ──
_found = sorted(glob.glob(str(INPUT_ROOT / "**/DisasterM3_Instruct/train_release.json"), recursive=True))
assert _found, "No manifest: attach the dataset abrarmohammedtanzim/disasterm3-mirror"
DATA_ROOT = Path(_found[0]).parent
_step0 = sorted(glob.glob(str(INPUT_ROOT / "**/layer3-step0/pairs.csv"), recursive=True))
assert _step0, "No pairs.csv: attach the output of your step 0 notebook (layer3-step0-pairs-and-checks, its full run)"
STEP0_DIR = Path(_step0[0]).parent

# ── A checkpoint from an earlier session: this notebook's output, attached as an input ──
_resume = sorted(glob.glob(str(INPUT_ROOT / "**/layer3-step2a/checkpoints/last.pt"), recursive=True))
assert len(_resume) <= 1, f"More than one earlier checkpoint is attached: {_resume}. Attach only the latest session's output."
RESUME = Path(_resume[0]) if _resume else None
assert not (SMOKE_TEST and RESUME), "The smoke test starts fresh: remove the attached step 2a output"

assert torch.cuda.is_available(), "No GPU: Settings → Accelerator → GPU T4 x2"
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True      # the training crops are all 512 x 512
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
(OUT_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
T_START = time.time()
print(f"✓ Layer 3 step 2a{'  [SMOKE TEST]' if SMOKE_TEST else ''}: pix2pix, {EPOCHS_RUN} epochs"
      f"{f' of {SMOKE_PAIRS_PER_EPOCH} pairs' if SMOKE_TEST else ''}, {CROP} px crops, seed {SEED}")
print(f"  Data: {DATA_ROOT}")
print(f"  Step 0's pairs: {STEP0_DIR / 'pairs.csv'}")
print(f"  Earlier checkpoint: {RESUME or 'none, so training starts fresh'}")
print(f"  Device: {torch.cuda.get_device_name(0)}; data loader workers: {NUM_WORKERS}; outputs: {OUT_DIR}")

✓ Layer 3 step 2a: pix2pix, 125 epochs, 512 px crops, seed 42
  Data: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct
  Step 0's pairs: /kaggle/input/notebooks/riditaafrin/layer3-step0-pairs-and-checks/layer3-step0/pairs.csv
  Earlier checkpoint: /kaggle/input/notebooks/riditaafrin/layer3-step2a-pix2pix-train/layer3-step2a/checkpoints/last.pt
  Device: Tesla T4; data loader workers: 4; outputs: /kaggle/working/layer3-step2a


## 1. The pairs: step 0's training rows, checked, and the 12 sample pairs
Cell 4 reads step 0's `pairs.csv` and finds each photo as R11's Cell 4 does. It checks that the training rows hold no validation photo, by name, by path, and by step 0's pixel hashes. It then picks the 12 sample pairs.

In [3]:
# ── Cell 4: Step 0's pairs: the training rows, their checks, and the 12 sample pairs ──
import cv2
import pandas as pd

cv2.setNumThreads(0)    # the data loader's workers decode photos in parallel instead

P0 = pd.read_csv(STEP0_DIR / "pairs.csv")
S0 = json.loads((STEP0_DIR / "summary.json").read_text(encoding="utf-8"))

# ── Path resolution (R11's Cell 4, word for word) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

def photo_path(name):
    """A photo named in pairs.csv. The manifest lists every photo under train_images/; resolved as in R11's Cell 4."""
    return resolve_path(f"train_images/{name}")

TRAIN = P0[P0["split"] == "train"].reset_index(drop=True)
VAL = P0[P0["split"] == "val"].reset_index(drop=True)
VAL["clean"] = VAL["clean"].eq(True)
CLEAN = VAL[VAL["clean"]]
TRAIN_PATHS = [(photo_path(a), photo_path(b)) for a, b in zip(TRAIN["after_image"], TRAIN["before_image"])]
VAL_PATHS = {p for a, b in zip(VAL["after_image"], VAL["before_image"]) for p in (photo_path(a), photo_path(b))}

def _names(df):
    return set(df["after_image"]) | set(df["before_image"])

_train_md5 = set(TRAIN["after_pixel_md5"]) | set(TRAIN["before_pixel_md5"])
_shares = VAL["after_pixel_md5"].isin(_train_md5) | VAL["before_pixel_md5"].isin(_train_md5)
N_VAL_SHARING = int(_shares.sum())     # E5's 40: their before-photo is also a training photo; none of them is clean
STEP0_FINDINGS = {
    "step 0's full run, every check passed": not S0["smoke_test"] and all(v == "passed" for v in S0["checks"].values()),
    "6,443 pairs: 5,798 training, 645 validation": (len(P0), len(TRAIN), len(VAL)) == (6443, 5798, 645),
    "605 clean": len(CLEAN) == 605,
    "every photo 1024 x 1024": bool((P0["height"].eq(1024) & P0["width"].eq(1024) & P0["before_same_size"].eq(True)).all()),
}
TRAIN_FINDINGS = {
    "every training photo found": all(p is not None for pair in TRAIN_PATHS for p in pair),
    "every validation photo found": None not in VAL_PATHS,
    "no validation photo among the training photos, by name": not (_names(TRAIN) & _names(VAL)),
    "no validation photo among the training photos, by path": not ({p for pair in TRAIN_PATHS for p in pair} & VAL_PATHS),
    "no photo of a clean validation pair among the training photos, by pixel hash": not (_shares & VAL["clean"]).any(),
}
TRAIN_NAMES_SHA256 = hashlib.sha256("\n".join(
    f"{a}\t{b}" for a, b in zip(TRAIN["after_image"], TRAIN["before_image"])).encode()).hexdigest()

# ── The 12 sample pairs: clean validation pairs, at random (seed 42); where an event has enough, with destroyed buildings ──
_rng = np.random.default_rng(SEED)

def _pick(g, n):
    with_destroyed = g[g["destroyed_px"] > 0]
    g = with_destroyed if len(with_destroyed) >= n else g
    return g.iloc[np.sort(_rng.choice(len(g), n, replace=False))]

_other_events = sorted(set(CLEAN["event"]) - {"turkey_earthquake"})
_chosen = sorted(_other_events[i] for i in _rng.choice(len(_other_events), SAMPLE_OTHER_EVENTS, replace=False))
SAMPLE = pd.concat([_pick(CLEAN[CLEAN["event"] == "turkey_earthquake"], SAMPLE_TURKEY)]
                   + [_pick(CLEAN[CLEAN["event"] == e], 1) for e in _chosen]).reset_index(drop=True)

for name, findings in (("Step 0's pairs.csv", STEP0_FINDINGS), ("Training rows", TRAIN_FINDINGS)):
    print(f"{'✓' if all(findings.values()) else '✗'} {name}: "
          + "; ".join(f"{k}: {'yes' if v else 'NO'}" for k, v in findings.items()))
print(f"  {N_VAL_SHARING} validation pairs share a photo with training by pixel hash, "
      f"{int((_shares & VAL['clean']).sum())} of them clean (E5: 40, none clean)")
print(f"  Sample pairs ({len(SAMPLE)}): " + ", ".join(f"{r.event} {r.pair_index}" for r in SAMPLE.itertuples()))

✓ Step 0's pairs.csv: step 0's full run, every check passed: yes; 6,443 pairs: 5,798 training, 645 validation: yes; 605 clean: yes; every photo 1024 x 1024: yes
✓ Training rows: every training photo found: yes; every validation photo found: yes; no validation photo among the training photos, by name: yes; no validation photo among the training photos, by path: yes; no photo of a clean validation pair among the training photos, by pixel hash: yes
  40 validation pairs share a photo with training by pixel hash, 0 of them clean (E5: 40, none clean)
  Sample pairs (12): turkey_earthquake 4657, turkey_earthquake 4694, beriut_explosion 31, hawaii_wildfire 65, hurricane_harvey 6197, kalehe_flooding 1193, midwest_flooding 6227, moore_tornado 1737, palu_tsunami 2227, portugal_wildfire 3422, santa_rosa_wildfire 3692, socal_fire 4060


## 2. The networks, the training crops and the picture distances
- Cell 6 holds pix2pix's generator, discriminator, initialisation and GAN loss, from the authors' code, kept to the options pix2pix uses by default. It checks their sizes against the reference's printout (54.414 M and 2.769 M parameters). It also defines the training crops.
- Cell 7 sets up PSNR, SSIM (scikit-image, with the settings of Wang et al. 2004) and LPIPS (AlexNet, version 0.1).

In [4]:
# ── Cell 6: pix2pix's networks and loss (the authors' code), and the training crops ──
import functools
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init

# From junyanz/pytorch-CycleGAN-and-pix2pix, models/networks.py (BSD licence): UnetGenerator, UnetSkipConnectionBlock,
# NLayerDiscriminator, the "normal" initialisation (gain 0.02) and GANLoss, kept to the options pix2pix uses by default.
NORM = functools.partial(nn.BatchNorm2d, affine=True, track_running_stats=True)   # the reference's norm "batch"

class UnetGenerator(nn.Module):
    """Create a Unet-based generator"""
    def __init__(self, input_nc, output_nc, num_downs, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetGenerator, self).__init__()
        # construct unet structure
        unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=None, norm_layer=norm_layer, innermost=True)  # add the innermost layer
        for i in range(num_downs - 5):          # add intermediate layers with ngf * 8 filters
            unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer, use_dropout=use_dropout)
        # gradually reduce the number of filters from ngf * 8 to ngf
        unet_block = UnetSkipConnectionBlock(ngf * 4, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf * 2, ngf * 4, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf, ngf * 2, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        self.model = UnetSkipConnectionBlock(output_nc, ngf, input_nc=input_nc, submodule=unet_block, outermost=True, norm_layer=norm_layer)  # add the outermost layer

    def forward(self, input):
        """Standard forward"""
        return self.model(input)


class UnetSkipConnectionBlock(nn.Module):
    """Defines the Unet submodule with skip connection.
        X -------------------identity----------------------
        |-- downsampling -- |submodule| -- upsampling --|
    """
    def __init__(self, outer_nc, inner_nc, input_nc=None,
                 submodule=None, outermost=False, innermost=False, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetSkipConnectionBlock, self).__init__()
        self.outermost = outermost
        if type(norm_layer) == functools.partial:
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d
        if input_nc is None:
            input_nc = outer_nc
        downconv = nn.Conv2d(input_nc, inner_nc, kernel_size=4,
                             stride=2, padding=1, bias=use_bias)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = norm_layer(inner_nc)
        uprelu = nn.ReLU(True)
        upnorm = norm_layer(outer_nc)

        if outermost:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc,
                                        kernel_size=4, stride=2,
                                        padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(inner_nc, outer_nc,
                                        kernel_size=4, stride=2,
                                        padding=1, bias=use_bias)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc,
                                        kernel_size=4, stride=2,
                                        padding=1, bias=use_bias)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]

            if use_dropout:
                model = down + [submodule] + up + [nn.Dropout(0.5)]
            else:
                model = down + [submodule] + up

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:   # add skip connections
            return torch.cat([x, self.model(x)], 1)


class NLayerDiscriminator(nn.Module):
    """Defines a PatchGAN discriminator"""
    def __init__(self, input_nc, ndf=64, n_layers=3, norm_layer=nn.BatchNorm2d):
        super(NLayerDiscriminator, self).__init__()
        if type(norm_layer) == functools.partial:  # no need to use bias as BatchNorm2d has affine parameters
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d

        kw = 4
        padw = 1
        sequence = [nn.Conv2d(input_nc, ndf, kernel_size=kw, stride=2, padding=padw), nn.LeakyReLU(0.2, True)]
        nf_mult = 1
        nf_mult_prev = 1
        for n in range(1, n_layers):  # gradually increase the number of filters
            nf_mult_prev = nf_mult
            nf_mult = min(2 ** n, 8)
            sequence += [
                nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=2, padding=padw, bias=use_bias),
                norm_layer(ndf * nf_mult),
                nn.LeakyReLU(0.2, True)
            ]

        nf_mult_prev = nf_mult
        nf_mult = min(2 ** n_layers, 8)
        sequence += [
            nn.Conv2d(ndf * nf_mult_prev, ndf * nf_mult, kernel_size=kw, stride=1, padding=padw, bias=use_bias),
            norm_layer(ndf * nf_mult),
            nn.LeakyReLU(0.2, True)
        ]

        sequence += [nn.Conv2d(ndf * nf_mult, 1, kernel_size=kw, stride=1, padding=padw)]  # output 1 channel prediction map
        self.model = nn.Sequential(*sequence)

    def forward(self, input):
        """Standard forward."""
        return self.model(input)


def init_weights(net, init_gain=0.02):
    """The reference's "normal" initialisation."""
    def init_func(m):  # define the initialization function
        classname = m.__class__.__name__
        if hasattr(m, 'weight') and (classname.find('Conv') != -1 or classname.find('Linear') != -1):
            init.normal_(m.weight.data, 0.0, init_gain)
            if hasattr(m, 'bias') and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find('BatchNorm2d') != -1:  # BatchNorm Layer's weight is not a matrix; only normal distribution applies.
            init.normal_(m.weight.data, 1.0, init_gain)
            init.constant_(m.bias.data, 0.0)
    net.apply(init_func)  # apply the initialization function <init_func>


class GANLoss(nn.Module):
    """The reference's GANLoss with gan_mode "vanilla": cross-entropy with logits against 1 (real) or 0 (generated)."""
    def __init__(self, target_real_label=1.0, target_fake_label=0.0):
        super(GANLoss, self).__init__()
        self.register_buffer('real_label', torch.tensor(target_real_label))
        self.register_buffer('fake_label', torch.tensor(target_fake_label))
        self.loss = nn.BCEWithLogitsLoss()

    def get_target_tensor(self, prediction, target_is_real):
        target_tensor = self.real_label if target_is_real else self.fake_label
        return target_tensor.expand_as(prediction)

    def __call__(self, prediction, target_is_real):
        prediction = prediction.float()          # ours: the loss in fp32 under mixed precision
        target_tensor = self.get_target_tensor(prediction, target_is_real)
        return self.loss(prediction, target_tensor)


def set_requires_grad(net, requires_grad):
    for param in net.parameters():
        param.requires_grad = requires_grad

# ── The training crops ──
def read_rgb(path):
    """A photo as RGB uint8, read as step 1 reads it: cv2, then BGR to RGB."""
    im = cv2.imread(str(path))
    if im is None:
        raise FileNotFoundError(f"Unreadable photo: {path}")
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

class PairCrops(torch.utils.data.Dataset):
    """Item i of an epoch: training pair i, with the same random crop, rotation and flip on both photos.
    They depend only on the seed, the epoch and i, so a second session draws what an unbroken run would."""
    def __init__(self, paths, epoch):
        self.paths, self.epoch = paths, epoch

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        after, before = (read_rgb(p) for p in self.paths[i])
        if after.shape != before.shape or min(after.shape[:2]) < CROP:
            raise ValueError(f"Pair {self.paths[i]}: sizes {after.shape} and {before.shape}")
        rng = np.random.default_rng([SEED, self.epoch, i])
        y = int(rng.integers(0, after.shape[0] - CROP + 1))
        x = int(rng.integers(0, after.shape[1] - CROP + 1))
        k, flip = int(rng.integers(4)), bool(rng.integers(2))

        def view(im):
            im = np.rot90(im[y:y + CROP, x:x + CROP], k)
            return torch.from_numpy(np.ascontiguousarray((im[:, ::-1] if flip else im).transpose(2, 0, 1)))
        return view(after), view(before), i

def to_unit(x):
    """uint8 pictures, (N, 3, H, W) tensors or one (H, W, 3) array, to the range -1..1 on the GPU."""
    if isinstance(x, np.ndarray):
        x = torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0)
    return x.to(DEVICE, non_blocking=True).float() / 127.5 - 1

def to_uint8(y):
    """The generator's output (1, 3, H, W), -1..1, to an RGB uint8 picture (H, W, 3)."""
    return ((y[0].float().clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).cpu().numpy()

# ── Self-checks: the reference's sizes, the output sizes, and the same crop on both photos ──
_g = UnetGenerator(3, 3, N_DOWNS, NGF, norm_layer=NORM, use_dropout=DROPOUT).to(DEVICE)
_d = NLayerDiscriminator(6, NDF, n_layers=D_LAYERS, norm_layer=NORM).to(DEVICE)
N_PARAMS = {"G": sum(p.numel() for p in _g.parameters()), "D": sum(p.numel() for p in _d.parameters())}
assert (round(N_PARAMS["G"] / 1e6, 3), round(N_PARAMS["D"] / 1e6, 3)) == (54.414, 2.769), N_PARAMS
with torch.no_grad():
    assert _g(torch.zeros(1, 3, CROP, CROP, device=DEVICE)).shape == (1, 3, CROP, CROP)
    assert _g(torch.zeros(1, 3, 1024, 1024, device=DEVICE)).shape == (1, 3, 1024, 1024)
    D_OUT = tuple(_d(torch.zeros(1, 6, CROP, CROP, device=DEVICE)).shape)
del _g, _d
_a, _b, _ = PairCrops([(TRAIN_PATHS[0][0], TRAIN_PATHS[0][0])], 0)[0]
assert _a.shape == (3, CROP, CROP) and _a.dtype == torch.uint8 and torch.equal(_a, _b), "the two photos got different crops"
_a1, _, _ = PairCrops([(TRAIN_PATHS[0][0], TRAIN_PATHS[0][0])], 0)[0]
assert torch.equal(_a, _a1), "the same seed, epoch and pair gave a different crop"
_x = np.random.default_rng(SEED).integers(0, 256, (8, 8, 3), dtype=np.uint8)
assert np.array_equal(to_uint8(to_unit(_x)), _x)
print(f"✓ pix2pix networks: generator {N_PARAMS['G']:,} parameters, discriminator {N_PARAMS['D']:,} "
      f"(the reference: 54.414 M and 2.769 M); discriminator output {D_OUT[2]} x {D_OUT[3]} patches on a {CROP} px crop")
print("✓ Training crops: the same crop, rotation and flip on both photos (self-checks passed)")

✓ pix2pix networks: generator 54,413,955 parameters, discriminator 2,768,705 (the reference: 54.414 M and 2.769 M); discriminator output 62 x 62 patches on a 512 px crop
✓ Training crops: the same crop, rotation and flip on both photos (self-checks passed)


In [5]:
# ── Cell 7: How close a picture is to the real before-photo: PSNR, SSIM and LPIPS ──
import lpips
from skimage.metrics import structural_similarity

LPIPS_NET = lpips.LPIPS(net="alex", version="0.1", verbose=False).to(DEVICE).eval()

def psnr(a, b):
    """Peak signal-to-noise ratio in dB, over all pixels and colours (uint8, peak 255)."""
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else float(10 * np.log10(255.0 ** 2 / mse))

def ssim(a, b):
    """SSIM with the settings of Wang et al. (2004): Gaussian window (sigma 1.5), each colour, averaged."""
    return float(structural_similarity(a, b, channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5,
                                       use_sample_covariance=False))

@torch.no_grad()
def lpips_distance(a, b):
    """LPIPS (AlexNet, version 0.1) at full size; 0 = the same picture."""
    return float(LPIPS_NET(to_unit(a), to_unit(b)).item())

def similarity(picture, real_before):
    return {"psnr": psnr(picture, real_before), "ssim": ssim(picture, real_before),
            "lpips": lpips_distance(picture, real_before)}

_p = read_rgb(TRAIN_PATHS[0][1])
_s = similarity(_p, _p)
assert _s["psnr"] == float("inf") and abs(_s["ssim"] - 1) < 1e-9 and abs(_s["lpips"]) < 1e-6, _s
_n = similarity(np.clip(_p.astype(int) + np.random.default_rng(SEED).integers(-20, 21, _p.shape), 0, 255).astype(np.uint8), _p)
assert 20 < _n["psnr"] < 40 and _n["ssim"] < 1 and _n["lpips"] > 0, _n
print(f"✓ PSNR, SSIM and LPIPS ready (self-checks passed; a photo with noise of ±20: {_n})")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 194MB/s]


✓ PSNR, SSIM and LPIPS ready (self-checks passed; a photo with noise of ±20: {'psnr': 26.678444103546433, 'ssim': 0.5516301906664032, 'lpips': 0.2852405309677124})


## 3. Training
Cell 9 trains for `EPOCHS` epochs, or until the session's time is up, and saves `checkpoints/last.pt` and `train_log.csv` after every epoch. With a checkpoint attached, it first loads it, after checking that it was made with the same settings and the same training pairs. In the smoke test, it saves, loads and compares after epoch 2, then proposes `EPOCHS` for the full run.

In [6]:
# ── Cell 9: Training, with a checkpoint after every epoch; a later session continues from it ──
import shutil
from torch.utils.data import DataLoader

CKPT = OUT_DIR / "checkpoints" / "last.pt"
AMP_ON = AMP and DEVICE.type == "cuda"
GAN_LOSS = GANLoss().to(DEVICE)
LOSS_NAMES = ["loss_D_real", "loss_D_fake", "loss_G_GAN", "loss_G_L1"]
CONFIG = {"smoke_test": SMOKE_TEST, "epochs": EPOCHS_RUN, "pairs_per_epoch": SMOKE_PAIRS_PER_EPOCH if SMOKE_TEST else len(TRAIN_PATHS),
          "crop": CROP, "batch_size": BATCH_SIZE, "ngf": NGF, "ndf": NDF, "n_downs": N_DOWNS, "d_layers": D_LAYERS,
          "dropout": DROPOUT, "lambda_l1": LAMBDA_L1, "lr": LR, "beta1": BETA1, "beta2": BETA2, "amp": AMP, "seed": SEED,
          "train_pairs": len(TRAIN_PATHS), "train_names_sha256": TRAIN_NAMES_SHA256}

def lr_at(epoch):
    """Epochs 1 to EPOCHS: the first half at LR, then linearly toward 0 (the reference's "linear" policy)."""
    n_decay = EPOCHS_RUN // 2
    return LR * (1.0 - max(0, epoch - (EPOCHS_RUN - n_decay)) / float(n_decay + 1))

def epoch_order(epoch):
    """The training pairs of an epoch, in a random order from the seed and the epoch (smoke test: the first 200)."""
    order = np.random.default_rng([SEED, epoch]).permutation(len(TRAIN_PATHS))
    return order[:SMOKE_PAIRS_PER_EPOCH] if SMOKE_TEST else order

def make_models():
    G = UnetGenerator(3, 3, N_DOWNS, NGF, norm_layer=NORM, use_dropout=DROPOUT)
    D = NLayerDiscriminator(6, NDF, n_layers=D_LAYERS, norm_layer=NORM)
    init_weights(G)
    init_weights(D)
    G, D = G.to(DEVICE), D.to(DEVICE)
    return {"G": G, "D": D,
            "opt_G": torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, BETA2)),
            "opt_D": torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, BETA2)),
            "scaler_G": torch.amp.GradScaler("cuda", enabled=AMP_ON),
            "scaler_D": torch.amp.GradScaler("cuda", enabled=AMP_ON)}

def save_checkpoint(M, epoch, history):
    tmp = CKPT.with_name("last.pt.tmp")
    torch.save({"config": CONFIG, "epoch": epoch, "history": history,
                "state": {k: v.state_dict() for k, v in M.items()},
                "rng": {"torch": torch.get_rng_state(),
                        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                        "numpy": np.random.get_state(), "python": random.getstate()}}, tmp)
    os.replace(tmp, CKPT)        # a session stopped while saving keeps the previous checkpoint

def load_checkpoint(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    diff = {k: (ck["config"].get(k), v) for k, v in CONFIG.items() if ck["config"].get(k) != v}
    if diff:
        raise RuntimeError(f"{path} was made with other settings or pairs, (checkpoint, now): {diff}")
    M = make_models()
    for k, v in M.items():
        v.load_state_dict(ck["state"][k])
    torch.set_rng_state(ck["rng"]["torch"])
    for i, state in enumerate(ck["rng"]["cuda"][:torch.cuda.device_count()]):   # T4 x2 saves two; any GPU count loads
        torch.cuda.set_rng_state(state, i)
    np.random.set_state(ck["rng"]["numpy"])
    random.setstate(ck["rng"]["python"])
    return M, ck["epoch"], ck["history"]

def _equal(a, b):
    if isinstance(a, torch.Tensor):
        return isinstance(b, torch.Tensor) and a.shape == b.shape and torch.equal(a.cpu(), b.cpu())
    if isinstance(a, dict):
        return isinstance(b, dict) and a.keys() == b.keys() and all(_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)):
        return isinstance(b, (list, tuple)) and len(a) == len(b) and all(_equal(x, y) for x, y in zip(a, b))
    return a == b

def train_epoch(M, epoch):
    """One epoch, as the reference's optimize_parameters(): the discriminator's step, then the generator's."""
    G, D, opt_G, opt_D, scaler_G, scaler_D = (M[k] for k in ("G", "D", "opt_G", "opt_D", "scaler_G", "scaler_D"))
    lr = lr_at(epoch)
    for opt in (opt_G, opt_D):
        for group in opt.param_groups:
            group["lr"] = lr
    G.train()
    D.train()
    order = epoch_order(epoch)
    loader = DataLoader(PairCrops(TRAIN_PATHS, epoch), batch_size=BATCH_SIZE, sampler=order.tolist(),
                        num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=4 if NUM_WORKERS else None)
    sums = torch.zeros(len(LOSS_NAMES), device=DEVICE, dtype=torch.float64)
    nonfinite = torch.zeros((), device=DEVICE, dtype=torch.int64)
    seen, steps, t0 = [], 0, time.time()
    for after, before, idx in loader:
        real_A, real_B = to_unit(after), to_unit(before)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ON):
            fake_B = G(real_A)
        # the discriminator: the real before-photo against the generated one, each beside the after-photo
        set_requires_grad(D, True)
        opt_D.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ON):
            loss_D_fake = GAN_LOSS(D(torch.cat((real_A, fake_B.detach()), 1)), False)
            loss_D_real = GAN_LOSS(D(torch.cat((real_A, real_B), 1)), True)
        loss_D = (loss_D_fake + loss_D_real) * 0.5
        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()
        # the generator: fool the discriminator, and stay close to the real before-photo (L1, weight 100)
        set_requires_grad(D, False)
        opt_G.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ON):
            loss_G_GAN = GAN_LOSS(D(torch.cat((real_A, fake_B), 1)), True)
        loss_G_L1 = F.l1_loss(fake_B.float(), real_B)
        loss_G = loss_G_GAN + loss_G_L1 * LAMBDA_L1
        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()
        batch = torch.stack([loss_D_real.detach(), loss_D_fake.detach(), loss_G_GAN.detach(), loss_G_L1.detach()]).double()
        nonfinite += (~torch.isfinite(batch)).any().long()
        sums += torch.nan_to_num(batch, nan=0.0, posinf=0.0, neginf=0.0)
        seen.extend(idx.tolist())
        steps += 1
    means = (sums / max(steps, 1)).tolist()
    return {"epoch": epoch, "lr": lr, "pairs": len(seen), "all_pairs_once": sorted(seen) == sorted(order.tolist()),
            **{k: round(v, 6) for k, v in zip(LOSS_NAMES, means)}, "nonfinite_steps": int(nonfinite.item()),
            "minutes": round((time.time() - t0) / 60, 3)}

if RESUME is not None:
    M, _done, HISTORY = load_checkpoint(RESUME)
    SESSION = max(h["session"] for h in HISTORY) + 1
    print(f"✓ Continuing from {RESUME}: {_done} of {EPOCHS_RUN} epochs done; this is session {SESSION}")
else:
    torch.manual_seed(SEED)
    M, _done, HISTORY, SESSION = make_models(), 0, [], 1
    print(f"✓ Starting fresh: session 1, {EPOCHS_RUN} epochs to train")

RELOAD_OK, SAVE_MINUTES, STOPPED_FOR_TIME = None, None, False
for epoch in range(_done + 1, EPOCHS_RUN + 1):
    if HISTORY:      # the next epoch is estimated from the last one, plus 5% and its save
        next_hours = (HISTORY[-1]["minutes"] * 1.05 + (SAVE_MINUTES if SAVE_MINUTES is not None else 3.0)) / 60
        if (time.time() - T_START) / 3600 + next_hours > SESSION_STOP_HOURS:
            STOPPED_FOR_TIME = True
            break
    row = train_epoch(M, epoch)
    row["session"] = SESSION
    HISTORY.append(row)
    _t = time.time()
    save_checkpoint(M, epoch, HISTORY)
    SAVE_MINUTES = (time.time() - _t) / 60
    pd.DataFrame(HISTORY).to_csv(OUT_DIR / "train_log.csv", index=False)
    print(f"  epoch {epoch}/{EPOCHS_RUN}: lr {row['lr']:.2e}; D real {row['loss_D_real']:.4f}, D fake {row['loss_D_fake']:.4f}, "
          f"G GAN {row['loss_G_GAN']:.4f}, G L1 {row['loss_G_L1']:.4f}; {row['minutes']:.1f} min + {SAVE_MINUTES:.1f} min saving; "
          f"{(time.time() - T_START) / 3600:.2f} h since Cell 2" + (f"; {row['nonfinite_steps']} steps NOT FINITE" if row["nonfinite_steps"] else ""))
    if row["nonfinite_steps"] > 0.01 * row["pairs"]:
        raise RuntimeError(f"STOP: training broke down: {row['nonfinite_steps']} steps with a loss that is not finite in epoch {epoch}")
    if SMOKE_TEST and epoch == SMOKE_RELOAD_AFTER:     # what a second session does: load the checkpoint and go on
        M2, _e2, _h2 = load_checkpoint(CKPT)
        RELOAD_OK = _e2 == epoch and _h2 == HISTORY and all(_equal(M[k].state_dict(), M2[k].state_dict()) for k in M)
        M = M2
        print(f"  {'✓' if RELOAD_OK else '✗'} Checkpoint saved, loaded into new networks and compared: "
              f"{'every weight and optimizer value the same' if RELOAD_OK else 'NOT THE SAME'}; epoch {epoch + 1} trains from the loaded copy")

if RESUME is not None and not CKPT.exists():     # no epoch trained here: keep the latest checkpoint in this output too
    shutil.copy2(RESUME, CKPT)
EPOCHS_DONE = HISTORY[-1]["epoch"] if HISTORY else 0
TRAINING_DONE = EPOCHS_DONE == EPOCHS_RUN
EPOCHS_PROPOSED = None
if SMOKE_TEST:
    _timed = [h for h in HISTORY if h["epoch"] >= 2]
    SEC_PER_PAIR = sum(h["minutes"] for h in _timed) * 60 / sum(h["pairs"] for h in _timed)
    MIN_PER_EPOCH = SEC_PER_PAIR * len(TRAIN_PATHS) / 60 + SAVE_MINUTES
    EPOCHS_PROPOSED = min(EPOCHS_CAP, int(TRAIN_HOURS * 60 // MIN_PER_EPOCH))
    print(f"\nTime per training pair (epochs 2 and 3): {SEC_PER_PAIR:.3f} s, so one epoch of {len(TRAIN_PATHS):,} pairs "
          f"takes about {MIN_PER_EPOCH:.1f} min with its save.")
    print(f"➜ For the full run, set EPOCHS = {EPOCHS_PROPOSED} in Cell 2: about {EPOCHS_PROPOSED * MIN_PER_EPOCH / 60:.1f} h "
          f"of training, in two sessions at most (the limit: {TRAIN_HOURS:.0f} h, and {EPOCHS_CAP} epochs). "
          f"One session alone would hold about {int(10 * 60 // MIN_PER_EPOCH)} epochs.")
elif STOPPED_FOR_TIME:
    print(f"\nSession {SESSION} stopped after epoch {EPOCHS_DONE} of {EPOCHS_RUN}, "
          f"{(time.time() - T_START) / 3600:.2f} h since Cell 2: the next epoch would end after {SESSION_STOP_HOURS} h.")
else:
    print(f"\n✓ Training done: {EPOCHS_DONE} epochs in {len({h['session'] for h in HISTORY})} session(s)")

✓ Continuing from /kaggle/input/notebooks/riditaafrin/layer3-step2a-pix2pix-train/layer3-step2a/checkpoints/last.pt: 106 of 125 epochs done; this is session 2
  epoch 107/125: lr 6.03e-05; D real 0.3775, D fake 0.3368, G GAN 2.1071, G L1 0.1678; 6.4 min + 0.0 min saving; 0.19 h since Cell 2
  epoch 108/125: lr 5.71e-05; D real 0.3760, D fake 0.3400, G GAN 2.0920, G L1 0.1666; 6.0 min + 0.0 min saving; 0.30 h since Cell 2
  epoch 109/125: lr 5.40e-05; D real 0.3755, D fake 0.3378, G GAN 2.0967, G L1 0.1670; 6.0 min + 0.0 min saving; 0.40 h since Cell 2
  epoch 110/125: lr 5.08e-05; D real 0.3714, D fake 0.3337, G GAN 2.1090, G L1 0.1669; 6.0 min + 0.0 min saving; 0.50 h since Cell 2
  epoch 111/125: lr 4.76e-05; D real 0.3739, D fake 0.3351, G GAN 2.0973, G L1 0.1667; 6.0 min + 0.0 min saving; 0.60 h since Cell 2
  epoch 112/125: lr 4.44e-05; D real 0.3712, D fake 0.3343, G GAN 2.0943, G L1 0.1665; 6.0 min + 0.0 min saving; 0.70 h since Cell 2
  epoch 113/125: lr 4.13e-05; D real 0.3713

## 4. The generated before-photos, how close they are, and the sample sheet
Cell 11 runs only when all epochs are done.
- It saves the generator alone (`checkpoints/generator_final.pt`).
- It then generates a before-photo for every validation pair (smoke test: the 12 sample pairs), from the full 1024 × 1024 after-photo.
- Each picture is saved as a PNG named like its after-photo, read back as step 1 reads it, and measured against the real before-photo. The after-photo is measured too (`copy`).
- The PNGs are packed into `generated_before_photos.zip`, with a note that they are generated, not photos.
- Last comes the sample sheet, `samples.jpg`. Its middle picture in each group of three is generated, not a photo, and says so.

In [7]:
# ── Cell 11: The generated before-photos, PSNR / SSIM / LPIPS, the zip and the sample sheet ──
import shutil
import zipfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

GEN_DIR = TMP_DIR / "generated"
ZIP_PATH = OUT_DIR / "generated_before_photos.zip"
FINAL = OUT_DIR / "checkpoints" / "generator_final.pt"
ZIP_NOTE = ("These pictures are GENERATED, NOT PHOTOS. Layer 3 step 2a: a pix2pix generator made each one from the "
            "after-photo of the same name (DisasterM3 validation pairs). Never use them as ground truth or show them as "
            "real photos.\n")
SIM_ROWS, GEN = [], {}
if TRAINING_DONE:
    G = M["G"]
    torch.save({"config": CONFIG, "epochs": EPOCHS_DONE, "generator": G.state_dict()}, FINAL)
    GEN["generator_final_sha256"] = sha256(FINAL)
    GEN_PAIRS = SAMPLE if SMOKE_TEST else VAL
    shutil.rmtree(GEN_DIR, ignore_errors=True)
    GEN_DIR.mkdir(parents=True)
    G.train()     # as the paper and the reference's test script: dropout on, batch norm with the picture's own statistics
    _t = time.time()
    for r in GEN_PAIRS.itertuples():
        after, before = read_rgb(photo_path(r.after_image)), read_rgb(photo_path(r.before_image))
        torch.manual_seed(SEED + int(r.pair_index))      # this picture's dropout
        with torch.no_grad():
            gen = to_uint8(G(to_unit(after)))
        out = GEN_DIR / r.after_image
        cv2.imwrite(str(out), cv2.cvtColor(gen, cv2.COLOR_RGB2BGR))
        back = cv2.imread(str(out))           # as step 1's before_photo() reads a folder: cv2, then BGR to RGB
        back = None if back is None else cv2.cvtColor(back, cv2.COLOR_BGR2RGB)
        SIM_ROWS.append({"pair_index": int(r.pair_index), "after_image": r.after_image, "before_image": r.before_image,
                         "event": r.event, "source": r.source, "clean": bool(r.clean),
                         "shape_ok": gen.shape == after.shape == (1024, 1024, 3),
                         "reads_back_exactly": back is not None and np.array_equal(back, gen),
                         **{f"generated_{k}": v for k, v in similarity(gen, before).items()},
                         **{f"copy_{k}": v for k, v in similarity(after, before).items()}})
    GEN["minutes_generating"] = round((time.time() - _t) / 60, 1)
    # ── The zip: the PNGs uncompressed (they are compressed already), and the note ──
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_STORED) as z:
        z.writestr("README.txt", ZIP_NOTE)
        for r in GEN_PAIRS.itertuples():
            z.write(GEN_DIR / r.after_image, arcname=r.after_image)
    with zipfile.ZipFile(ZIP_PATH) as z:
        GEN["zip_pngs"] = sorted(n for n in z.namelist() if n.endswith(".png"))
        GEN["zip_test"] = z.testzip()         # None = every file's checksum is right
    GEN["zip_sha256"] = sha256(ZIP_PATH)
    GEN["zip_mb"] = round(ZIP_PATH.stat().st_size / 2 ** 20, 1)
    # ── The sample sheet: 12 pairs, two columns of six groups: after-photo, generated, real before-photo ──
    fig, axes = plt.subplots(6, 6, figsize=(15.6, 18.6), dpi=100)
    for k, r in enumerate(SAMPLE.itertuples()):
        tiles = [(read_rgb(photo_path(r.after_image)), f"{r.event} ({r.pair_index})\nafter-photo", "black"),
                 (read_rgb(GEN_DIR / r.after_image), "GENERATED, NOT A PHOTO\n(pix2pix, from the after-photo)", "crimson"),
                 (read_rgb(photo_path(r.before_image)), "real before-photo\n", "black")]
        for j, (im, label, colour) in enumerate(tiles):
            ax = axes[k % 6, (k // 6) * 3 + j]
            ax.imshow(cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA))
            ax.set_title(label, fontsize=8, color=colour)
            ax.set_xticks([])
            ax.set_yticks([])
            if j == 1:
                for side in ax.spines.values():
                    side.set_edgecolor("crimson")
                    side.set_linewidth(2)
    _steps = sum(h["pairs"] for h in HISTORY)
    fig.suptitle(f"Layer 3 step 2a{f' [SMOKE TEST: {_steps} training steps]' if SMOKE_TEST else ''}: pix2pix on 12 clean validation "
                 f"pairs, after {EPOCHS_DONE} epochs. In each group of three, the middle picture is generated, not a photo.",
                 fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(OUT_DIR / "samples.jpg", dpi=100, pil_kwargs={"quality": 90})
    plt.close(fig)
    shutil.rmtree(GEN_DIR)                    # the zip holds them
    print(f"✓ Generated {len(SIM_ROWS)} before-photos in {GEN['minutes_generating']} min; zip {GEN['zip_mb']} MB; samples.jpg written")
else:
    print(f"Training is not finished ({EPOCHS_DONE} of {EPOCHS_RUN} epochs), so nothing is generated in this session.")

✓ Generated 645 before-photos in 17.2 min; zip 1092.2 MB; samples.jpg written


## 5. The files, the checks and the printout
Cell 13 writes `similarity_per_pair.csv` (one row per generated picture), `similarity.csv` (the means: on all pairs and on the clean ones; overall, for Turkey, for the rest, and for each event) and `summary.json`. It then prints the checks. A session that stops before the last epoch checks only 1 and 2, and says what to do next.

In [8]:
# ── Cell 13: summary.json, similarity.csv, the checks and the printout ──
import importlib.metadata as _md
import platform
import skimage
import torchvision

SIM_COLS = [f"{w}_{m}" for w in ("generated", "copy") for m in ("psnr", "ssim", "lpips")]
SIM = pd.DataFrame(SIM_ROWS)
SIM_TABLE = pd.DataFrame()
if TRAINING_DONE:
    SIM.to_csv(OUT_DIR / "similarity_per_pair.csv", index=False)
    _rows = []
    for pairs_name, gp in (("all", SIM), ("clean", SIM[SIM["clean"]])):
        turkey = gp["event"].str.startswith("turkey")
        groups = [("all", gp), ("Turkey", gp[turkey]), ("not Turkey", gp[~turkey])]
        groups += [(f"event: {e}", g) for e, g in gp.groupby("event")]
        for group, g in groups:
            if len(g):
                _rows.append({"pairs": pairs_name, "group": group, "n_pairs": len(g),
                              **{c: round(float(g[c].mean()), 6) for c in SIM_COLS}})
    SIM_TABLE = pd.DataFrame(_rows)
    SIM_TABLE.to_csv(OUT_DIR / "similarity.csv", index=False)

# ── The checks ──
_pairs_ok = all(h["all_pairs_once"] and h["pairs"] == CONFIG["pairs_per_epoch"] for h in HISTORY)
CHECKS = [
    ("1. Step 0's pairs.csv: its full run; 6,443 pairs, 5,798 / 645, 605 clean; 1024 x 1024", all(STEP0_FINDINGS.values()),
     "; ".join(k for k, v in STEP0_FINDINGS.items() if not v) or "as expected"),
    ("2. Training sees only training photos", all(TRAIN_FINDINGS.values()) and _pairs_ok,
     "; ".join([k for k, v in TRAIN_FINDINGS.items() if not v] + ([] if _pairs_ok else ["an epoch's pairs are not as planned"]))
     or f"{len(HISTORY)} epochs of {CONFIG['pairs_per_epoch']:,} training pairs, each once; "
        f"{N_VAL_SHARING} validation pairs share a photo with training, none of them clean"),
]
if TRAINING_DONE:
    _expected_n = len(SAMPLE) if SMOKE_TEST else 645
    _names_ok = GEN["zip_pngs"] == sorted(GEN_PAIRS["after_image"])
    _finite = bool(np.isfinite(SIM[SIM_COLS].to_numpy()).all())
    _key = "12" if SMOKE_TEST else "605"
    _judged = SIM if SMOKE_TEST else SIM[SIM["clean"]]
    COPY_FOUND = {m: float(_judged[f"copy_{m}"].mean()) for m in ("psnr", "ssim", "lpips")}
    _copy_ok = COPY_EXPECTED is not None and all(abs(COPY_FOUND[m] - COPY_EXPECTED[_key][m]) <= COPY_TOLERANCE[m] for m in COPY_FOUND)
    _bad_steps = sum(h["nonfinite_steps"] for h in HISTORY)
    _sheet = SAMPLE.assign(turkey=SAMPLE["event"].str.startswith("turkey"))
    _sheet_ok = (len(_sheet) == 12 and bool(_sheet["clean"].all()) and _sheet["event"].nunique() == 11
                 and int(_sheet["turkey"].sum()) == 2 and (OUT_DIR / "samples.jpg").exists())
    CHECKS += [
        ("3. Training ran all its epochs, every loss finite" + (", and the loaded checkpoint matched" if SMOKE_TEST else ""),
         TRAINING_DONE and _bad_steps == 0 and (RELOAD_OK is True or not SMOKE_TEST),
         f"{EPOCHS_DONE} of {EPOCHS_RUN} epochs; {_bad_steps} steps with a loss that is not finite"
         + (f"; reload {'matched' if RELOAD_OK else 'DID NOT MATCH'}" if SMOKE_TEST else "")),
        (f"4. All {_expected_n} pictures in the zip: 1024 x 1024 RGB PNG, named like the after-photos, reading back exactly",
         len(SIM) == _expected_n and _names_ok and GEN["zip_test"] is None and bool(SIM["shape_ok"].all())
         and bool(SIM["reads_back_exactly"].all()),
         f"{len(GEN['zip_pngs'])} PNGs; names {'as expected' if _names_ok else 'NOT as expected'}; zip test "
         f"{'passed' if GEN['zip_test'] is None else 'FAILED at ' + str(GEN['zip_test'])}; "
         f"{int(SIM['shape_ok'].sum())} of the right size; {int(SIM['reads_back_exactly'].sum())} read back exactly"),
        ("5. The sample sheet: 12 clean validation pairs from 11 events, 2 of them Turkey", _sheet_ok,
         f"{len(_sheet)} pairs, {int(_sheet['clean'].sum())} clean, {_sheet['event'].nunique()} events, {int(_sheet['turkey'].sum())} Turkey"),
        (f"6. PSNR, SSIM and LPIPS finite; copy matches the PC on the {_key} {'sample' if SMOKE_TEST else 'clean'} pairs", _finite and _copy_ok,
         f"all finite: {'yes' if _finite else 'NO'}; copy here " + ", ".join(f"{m} {COPY_FOUND[m]:.6f}" for m in COPY_FOUND)
         + (f"; on the PC " + ", ".join(f"{m} {COPY_EXPECTED[_key][m]:.6f}" for m in COPY_FOUND) if COPY_EXPECTED else "; no PC numbers")),
    ]

VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
            "numpy": np.__version__, "opencv": cv2.__version__, "pandas": pd.__version__, "scikit-image": skimage.__version__,
            "lpips": _md.version("lpips"), "matplotlib": matplotlib.__version__}
summary = {
    "task": "Layer 3 step 2a", "smoke_test": SMOKE_TEST, "session": SESSION,
    "status": "done" if TRAINING_DONE else f"stopped after epoch {EPOCHS_DONE} of {EPOCHS_RUN}: continue in the next session",
    "minutes_this_session": round((time.time() - T_START) / 60, 1), "device": torch.cuda.get_device_name(0),
    "versions": VERSIONS, "config": CONFIG, "parameters": N_PARAMS, "discriminator_output": D_OUT,
    "epochs_done": EPOCHS_DONE, "sessions": sorted({h["session"] for h in HISTORY}),
    "training_minutes": round(sum(h["minutes"] for h in HISTORY), 1),
    "minutes_per_epoch_mean": round(float(np.mean([h["minutes"] for h in HISTORY])), 2) if HISTORY else None,
    "smoke_proposal": {"seconds_per_pair": round(SEC_PER_PAIR, 4), "minutes_per_epoch": round(MIN_PER_EPOCH, 1),
                       "epochs_for_two_sessions": EPOCHS_PROPOSED, "train_hours": TRAIN_HOURS, "cap": EPOCHS_CAP} if SMOKE_TEST else None,
    "resumed_from": str(RESUME) if RESUME else None, "step0_pairs_csv": str(STEP0_DIR / "pairs.csv"),
    "val_pairs_sharing_a_photo_with_training": N_VAL_SHARING,
    "sample_pairs": SAMPLE[["pair_index", "after_image", "event"]].to_dict("records"),
    "generation": {k: v for k, v in GEN.items() if k != "zip_pngs"} | {"pictures": len(SIM_ROWS),
                   "mode": "train mode (dropout on, per-picture batch norm), dropout seeded 42 + pair_index, fp32"},
    "copy_expected_from_pc": COPY_EXPECTED,
    "similarity_clean": SIM_TABLE[SIM_TABLE["pairs"] == "clean"].head(3).to_dict("records") if len(SIM_TABLE) else None,
    "label": "Every generated picture is generated, not a photo.",
    "checks": {name: ("passed" if ok else "FAILED") for name, ok, _ in CHECKS},
}
(OUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)), encoding="utf-8")

# ── The printout ──
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
_tag = " [SMOKE TEST]" if SMOKE_TEST else ""
print(f"Layer 3 step 2a{_tag}: training, {EPOCHS_DONE} of {EPOCHS_RUN} epochs (session {SESSION})")
print(pd.DataFrame(HISTORY)[["epoch", "session", "lr", "pairs"] + LOSS_NAMES + ["nonfinite_steps", "minutes"]].to_string(index=False))
if TRAINING_DONE:
    print(f"\nHow close to the real before-photo: means over the {'12 sample' if SMOKE_TEST else '605 clean'} pairs "
          "(PSNR and SSIM: higher is closer; LPIPS: lower is closer)")
    print(SIM_TABLE[(SIM_TABLE["pairs"] == ("all" if SMOKE_TEST else "clean"))
                    & SIM_TABLE["group"].isin(["all", "Turkey", "not Turkey"])].to_string(index=False))
print(f"\nLayer 3 step 2a{_tag}: the checks")
for name, ok, found in CHECKS:
    print(f"  {'✓' if ok else '✗'} {name}: {'passed' if ok else 'FAILED'} ({found})")
_files = "summary.json, train_log.csv, checkpoints/last.pt" + (
    ", similarity_per_pair.csv, similarity.csv, samples.jpg, generated_before_photos.zip, checkpoints/generator_final.pt"
    if TRAINING_DONE else "")
print(f"\n✓ Wrote {OUT_DIR}: {_files} ({summary['minutes_this_session']} min since Cell 2)")
failed = [name for name, ok, _ in CHECKS if not ok]
if failed:
    raise RuntimeError("STOP: send this printout to Abrar. Failed: " + "; ".join(failed))
if SMOKE_TEST:
    print(f"✓ All six checks passed on the smoke test. For the full run: SMOKE_TEST = False and EPOCHS = {EPOCHS_PROPOSED}")
elif TRAINING_DONE:
    print("✓ All six checks passed. Step 2b reads the pictures in generated_before_photos.zip")
else:
    print(f"✓ Checks 1 and 2 passed. To continue: attach this version's output as an input (+ Add Input → Notebooks), "
          f"change nothing else, and Save & Run All. {EPOCHS_RUN - EPOCHS_DONE} epochs are left.")

Layer 3 step 2a: training, 125 of 125 epochs (session 2)
 epoch  session       lr  pairs  loss_D_real  loss_D_fake  loss_G_GAN  loss_G_L1  nonfinite_steps  minutes
     1        1 0.000200   5798     0.449348     0.384922    1.855472   0.207999                0    8.652
     2        1 0.000200   5798     0.452716     0.369776    1.843449   0.205141                0    5.968
     3        1 0.000200   5798     0.456085     0.378771    1.801724   0.202496                0    5.971
     4        1 0.000200   5798     0.457029     0.383158    1.782433   0.199270                0    5.968
     5        1 0.000200   5798     0.453693     0.380847    1.784285   0.196728                0    5.967
     6        1 0.000200   5798     0.446372     0.379279    1.825088   0.194337                0    5.973
     7        1 0.000200   5798     0.445243     0.379196    1.824125   0.192928                0    5.963
     8        1 0.000200   5798     0.445191     0.382793    1.833054   0.191106       